In [ ]:
# Data Cleaning Practice: [Python Pandas and Numpy]

## Source
# YouTube Tutorial: [Watch me Cleaning Data in minutes with Python](https://youtu.be/NeJKaolLQqU?si=KayVfu2wcfzpUYGU
# YouTube Channel: @loresowhat

## Objective
#Clean messy data using Python

## Techniques Practiced
#- Header cleaner
#- Type conversion and currency cleaning
#- Categorical typos
#- handling mixed booleans
#- Date parsing
#- Logical integrity (clicks vs impressions)
#- Logical integrity (time travel)
#- Handling ouliers (winsoring)
#- String parsing (feature extractions)

## Notes
# I really like @loresowhat approach to data cleaning.  I have use this approach to create my own style of data 
# cleaining with Python.  I have been using it to learn more about the pandas and numpy libraries. 

In [3]:
import os
print(os.listdir())

['data_cleaning_template.docx', '.DS_Store', 'cleaning_data_exercise_3-18-26.ipynb', 'Untitled.ipynb', '~$utube_lenzo_datacleaning_screenshots.docx', 'youtube_lenzo_datacleaning_screenshots.docx', 'marketing_campaign_data_messy.csv', '🧭 PHASE 1.docx', 'Cleaning_practice_3-18.ipynb', 'cleaning_exercise_3-20-26.ipynb', 'cleaning_exercise_3-21-26.ipynb', '.ipynb_checkpoints', 'Cleaned Data_Different_Types']


In [27]:
import pandas as pd
import numpy as np

# Load the messy data
df = pd.read_csv('marketing_campaign_data_messy.csv')

print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded Dataset: 2020 rows, 12 columns


In [28]:
# =========================================================
# STEP 1: HEADER CLEANING
# =========================================================

print(df.columns.tolist())

df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')

print("FIX APPLIED")
print(df.columns.tolist())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
FIX APPLIED
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


In [29]:
# =========================================================
# STEP 2: TYPE CONVERSION & CURRENCY CLEANING
# =========================================================

dirty_spend_mask = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask,['campaign_id','spend']].head(3))

df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
df['spend'] = pd.to_numeric(df['spend'])

print("FIX APPLIED")
print(df.loc[dirty_spend_mask,['campaign_id','spend']].head(3))


   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
FIX APPLIED
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


In [30]:
# =========================================================
# STEP 3: CATEGORICAL TYPOS (FUZZY LOGI)
# =========================================================

print(df['channel'].unique())

cleanup_map = {
    'E-mail': 'Email',
    'Gogle': 'Google Ads',
    'Tik_Tok': 'TikTok',
    'Facebok': 'Facebook',
    'Insta_gram': 'Instagram',
    'N/A': np.nan
}

df['channel'] = df['channel'].replace(cleanup_map)
print("FIX APPLIED")
print(df['channel'].unique())


['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']
FIX APPLIED
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan]


In [31]:
# =========================================================
# STEP 4: HANDLING MIXED BOOLEANS
# =========================================================

print(df['active'].unique())

bool_map = {
    'Y': True,
    '0': False,
    'No': False,
    'Yes': True,
    '1': True,
    0: False,
    1: True
}

df['active'] = df['active'].map(bool_map).fillna(False).astype(bool)

print("FIX APPLIED")
print(df['active'].unique())

    

['Y' '0' 'No' 'True' 'Yes' '1' 'False']
FIX APPLIED
[ True False]


/var/folders/5s/3j6jw5vj7r91hbj15ct1bfqc0000gn/T/ipykernel_3675/2441952206.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['active'] = df['active'].map(bool_map).fillna(False).astype(bool)


In [32]:
# =========================================================
# STEP 5: DATE PARSING
# =========================================================

print(df['start_date'].dtype)

df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')

print("FIX APPLIED")
print(df['start_date'].dtype)

object
FIX APPLIED
datetime64[ns]


/var/folders/5s/3j6jw5vj7r91hbj15ct1bfqc0000gn/T/ipykernel_3675/2165737808.py:8: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')


In [33]:
df = df.loc[:, ~df.columns.duplicated()]

In [34]:
# =========================================================
# STEP 6: LOGICAL INTEGRITY (CLICKS VS IIMPRESSIONS)
# =========================================================

impossible_mask = df['clicks'] > df['impressions']
print(df.loc[impossible_mask,['campaign_id','impressions','clicks']].head(3))

Empty DataFrame
Columns: [campaign_id, impressions, clicks]
Index: []


In [36]:
# =========================================================
# STEP 7: LOGICAL INTEGRITY (TIME TRAVEL)
# =========================================================

time_travel_mask = df['end_date'] < df['start_date']
print(df.loc[time_travel_mask,['campaign_id','start_date','end_date']].head(3))

df.loc[time_travel_mask, 'end_date'] = df.loc[time_travel_mask, 'start_date'] + pd.Timedelta(days=30)

print("FIX APPLIED")
print(df.loc[time_travel_mask,['campaign_id','start_date','end_date']].head(3))


   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-01
54   CMP-00055 2023-09-01 2023-08-27
71   CMP-00072 2023-02-01 2023-01-27
FIX APPLIED
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-06-05
54   CMP-00055 2023-09-01 2023-10-01
71   CMP-00072 2023-02-01 2023-03-03


In [39]:
# =========================================================
# STEP 8: HANDLING OUTLIERS (WINSORIZING)
# =========================================================

Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)

IQR = Q3 - Q1
upper_limit = Q3 + (3 * IQR)

outlier_mask = df['spend'] > upper_limit
print(df.loc[outlier_mask,['campaign_id','spend']].head(3))

print("FIX APPLIED")
df.loc[outlier_mask, 'spend'] = upper_limit
print(df.loc[outlier_mask,['campaign_id','spend']].head(3))

     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
FIX APPLIED
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375


In [42]:
# =========================================================
# STEP 9: HANDLING OUTLIERS 
# =========================================================

print(df['campaign_name'].head(3))

df['season'] = df['campaign_name'].str.extract(r'Q\d_([^_]+)_')

print("FIX APPLIED")

print(df[['campaign_name','season']].head(3))

0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: object
FIX APPLIED
         campaign_name  season
0  Q4_Summer_CMP-00001  Summer
1  Q1_Launch_CMP-00002  Launch
2  Q3_Winter_CMP-00003  Winter
